# Week 9 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-09.md`](../../weeks/week-09.md) · **Given engine:** [`strips.py`](strips.py)

Teaching walkthrough of the week-9 studio (STRIPS + LLM-Modulo, arm D). It **imports
the reference code from [`solution.py`](solution.py)** — never re-pastes it — so the
answer shown here is byte-for-byte the one the provided test grades. Correctness is
proved separately: `python3 studios/_verify_solutions.py week-09` runs the unmodified
`test_planning.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: put the week folder (for solution/strips) and the repo root
# (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-09"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import strips
from strips import blocks_problem, plan, verify
import solution

## Task 1 — formalize the GRIPPER domain

`gripper_actions(balls)` builds every ground `Action` (three fluent sets each), and
`gripper_problem(n)` assembles the init/goal. The GIVEN planner, heuristics, and
`verify()` then run on it **unchanged** — that reuse is the week's payoff. From
`solution.py`:

In [ ]:
print(inspect.getsource(solution.gripper_actions))
print(inspect.getsource(solution.gripper_problem))

In [ ]:
# Mirror test_gripper_domain_is_solvable_and_verifies.
p = solution.gripper_problem(2)
result, exp = plan(p, "add")
assert result is not None, "planner found no plan — check schemas"
ok, err = verify(result, p)
assert ok, err
print(f"gripper(2) solved in {len(result)} steps ({exp} expansions); verify() accepts it:")
for i, a in enumerate(result, 1):
    print(f"  {i}. {a}")

# Mirror test_gripper_goal_requires_moving_every_ball: wandering is not a solution.
ok, err = verify(["move(a,b)", "move(b,a)"], p)
assert not ok and "goal not reached" in err, (ok, err)
print("\nrobby just wandering ->", err)

## Task 2 — arm D: the verifier-in-the-loop

`refine_with_verifier(propose, problem, max_retries)` runs *propose → verify → feed
the error back → retry*. The correctness decision belongs entirely to the GIVEN
`verify()`; only the solver's error string is fed back (never a ground-truth answer).
From `solution.py`:

In [ ]:
print(inspect.getsource(solution.refine_with_verifier))

In [ ]:
# Mirror test_refinement_loop_converges: a flaky generator (wrong, incomplete,
# then right) is driven to a verified plan by the loop.
p = blocks_problem(3)
correct = [a.name for a in plan(p, "add")[0]]
def flaky(prob, attempt, last_error):
    if attempt == 0: return ["stack(a,b)"] + correct   # missing the pick-up
    if attempt == 1: return correct[:-1]               # stops one short
    return correct                                     # finally right

final, log = solution.refine_with_verifier(flaky, p, max_retries=3)
assert final is not None and verify(final, p)[0]
assert log[-1][1] is True and len(log) == 3, log
print("arm-D loop log (attempt, ok, error):")
for row in log:
    print("  ", row)
print(f"converged on attempt {len(log)} to a verified plan")

## The guarantee, watched live — a confident, wrong generator is never certified

**The thesis of the course, as a runnable assertion.** A generator that always returns
a legal-but-goal-missing plan — no matter what error we feed back — is *never* handed a
certificate, because correctness comes from the sound `verify()`, not from trusting the
proposal. Mirrors `test_unverified_wrong_plan_is_never_accepted`.

In [ ]:
def confidently_wrong(prob, attempt, last_error):
    return ["pick-up(a)", "put-down(a)"]      # applies fine, wrong goal, every time

final, log = solution.refine_with_verifier(confidently_wrong, p, max_retries=3)
print("log (attempt, ok, error):")
for row in log:
    print("  ", row)
assert final is None, "loop wrongly CERTIFIED a goal-missing plan"
assert all(ok is False for _, ok, _ in log)
print("\nGUARANTEE HOLDS: final =", final,
      "-- a confident-but-wrong generator is never certified. The guarantee lives in verify().")

## The LLM in the loop (headless demo)

In a real arm D, `propose` is an LLM (`aicourse.llm.LLM`). We run it headless with the
deterministic `echo` backend so the notebook executes anywhere.

> **The `echo` backend is a FAKE** — its text is not a model result. For a real run,
> re-run with `backend="ollama"` (or `"manual"`). The point stands regardless of the
> backend: **only `verify()` decides correctness**, and only its error is fed back —
> never the answer.

In [ ]:
from aicourse.llm import LLM
llm = LLM(backend="echo")   # echo = headless-safe FAKE

def make_llm_propose(problem):
    correct = [a.name for a in plan(problem, "add")[0]]
    def propose(prob, attempt, last_error):
        # A real generator would prompt the LLM with the problem + last_error:
        prompt = f"Blocks World plan; attempt {attempt}; previous error: {last_error}"
        _ = llm.complete(prompt)            # FAKE echo call, shown for shape only
        # For a reproducible notebook we return the wrong-then-right sequence.
        return ["stack(a,b)"] + correct if attempt == 0 else correct
    return propose

final, log = solution.refine_with_verifier(make_llm_propose(p), p, max_retries=3)
print("headless arm-D log:", log)
print("verified plan found:", final is not None, "(decided by verify(), not the echo text)")